In [1]:
import os
import shutil
import pandas as pd
import re
from tqdm import tqdm

# ==========================================
# CONFIGURARE CĂI
# ==========================================
# Folderul de unde luăm pozele (probabil rezultatul de la scriptul anterior)
INPUT_DIR = "../datasets/aptos_augmented_cropped" 

# Folderul final unde vom crea structura PyTorch-ready
OUTPUT_DIR = "../datasets/aptos_augmented_reorganized"

# Maparea foldere <-> fișiere CSV
SPLITS = {
    'train': {'folder': 'train_images', 'csv': 'train_1.csv'},
    'val':   {'folder': 'val_images',   'csv': 'valid.csv'},
    'test':  {'folder': 'test_images',  'csv': 'test.csv'}
}

# Clasele noastre pentru retinopatie
CLASSES = ['0', '1', '2', '3', '4']

In [2]:
def generate_smart_filename(orig_name, index):
    """
    Analizează numele vechi și generează un nume curat de tipul:
    img_00001_right_aug_2.png
    """
    orig_lower = orig_name.lower()
    
    # 1. Detectăm ochiul
    if 'right' in orig_lower:
        ochi = 'right'
    elif 'left' in orig_lower:
        ochi = 'left'
    else:
        ochi = 'unk' # unknown (dacă nu e specificat)
        
    # 2. Detectăm dacă este augmentată
    aug_match = re.search(r'aug_(\d+)', orig_lower)
    if aug_match:
        aug_str = f"_aug_{aug_match.group(1)}"
    elif 'aug' in orig_lower:
        aug_str = "_aug_x"
    else:
        aug_str = "" # Nu e augmentată
        
    # 3. Păstrăm extensia (.png, .jpeg)
    ext = os.path.splitext(orig_name)[1]
    if not ext:
        ext = '.png'
        
    # Construim numele final
    return f"img_{index:05d}_{ochi}{aug_str}{ext}"

def find_columns(df):
    """Găsește automat care coloană e numele imaginii și care e clasa în CSV."""
    img_col, label_col = None, None
    
    # Variante posibile pentru coloana cu numele imaginii
    for col in ['id_code', 'image', 'image_id']:
        if col in df.columns: img_col = col; break
            
    # Variante posibile pentru coloana cu clasa (0-4)
    for col in ['diagnosis', 'level', 'class', 'label']:
        if col in df.columns: label_col = col; break
            
    # Dacă nu le găsește după nume, presupunem că prima e imaginea și a doua clasa
    if not img_col: img_col = df.columns[0]
    if not label_col: label_col = df.columns[1]
        
    return img_col, label_col

def reorganize_dataset():
    print(f"🚀 Începem reorganizarea în format ImageFolder: {OUTPUT_DIR}\n")
    
    # Creăm structura principală de foldere
    for split_name in SPLITS.keys():
        for cls in CLASSES:
            os.makedirs(os.path.join(OUTPUT_DIR, split_name, cls), exist_ok=True)
            
    # Contor global pentru numerotarea imaginilor
    global_idx = 1
    
    # Procesăm fiecare split (train, val, test)
    for split_name, info in SPLITS.items():
        csv_path = os.path.join(INPUT_DIR, info['csv'])
        img_folder = os.path.join(INPUT_DIR, info['folder'])
        
        if not os.path.exists(csv_path) or not os.path.exists(img_folder):
            print(f"⚠️ Lipsesc datele pentru '{split_name}'. Sărim peste.")
            continue
            
        print(f"📂 Procesăm {split_name.upper()}...")
        
        # Citim CSV-ul
        df = pd.read_csv(csv_path)
        img_col, label_col = find_columns(df)
        
        # Iterăm prin fiecare rând din tabel
        for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Mutare {split_name}"):
            orig_name = str(row[img_col])
            label = str(row[label_col])
            
            # Adăugăm extensia dacă lipsește din CSV (în Kaggle APTOS de obicei lipsește '.png' din tabel)
            if not orig_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                orig_name += '.png'
                
            src_img_path = os.path.join(img_folder, orig_name)
            
            if os.path.exists(src_img_path):
                # Generăm noul nume sugestiv
                new_name = generate_smart_filename(orig_name, global_idx)
                
                # Calea finală: dataset_reorganized / train / 4 / img_00001_right.png
                dst_img_path = os.path.join(OUTPUT_DIR, split_name, label, new_name)
                
                # Copiem imaginea
                shutil.copy2(src_img_path, dst_img_path)
                global_idx += 1
            else:
                pass # Poți da un print aici dacă vrei să vezi ce poze lipsesc fizic de pe disc

    print(f"\n✅ Gata! Datele sunt structurate perfect în folderul: {OUTPUT_DIR}")

if __name__ == "__main__":
    reorganize_dataset()

🚀 Începem reorganizarea în format ImageFolder: ../datasets/aptos_augmented_reorganized

📂 Procesăm TRAIN...


Mutare train: 100%|██████████| 2930/2930 [00:05<00:00, 537.06it/s]


📂 Procesăm VAL...


Mutare val: 100%|██████████| 366/366 [00:00<00:00, 930.99it/s] 


📂 Procesăm TEST...


Mutare test: 100%|██████████| 366/366 [00:00<00:00, 840.72it/s]


✅ Gata! Datele sunt structurate perfect în folderul: ../datasets/aptos_augmented_reorganized


In [3]:
# Show number of images from each class in the reorganized dataset
def count_images_per_class():
    print(f"📊 Număr de imagini per clasă în {OUTPUT_DIR}:\n")
    for split_name in SPLITS.keys():
        print(f"--- {split_name.upper()} ---")
        for cls in CLASSES:
            cls_folder = os.path.join(OUTPUT_DIR, split_name, cls)
            if os.path.exists(cls_folder):
                num_images = len([f for f in os.listdir(cls_folder) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
                print(f"Clasa {cls}: {num_images} imagini")
            else:
                print(f"Clasa {cls}: 0 imagini (folder lipsă)")
        print()

if __name__ == "__main__":
    count_images_per_class()

📊 Număr de imagini per clasă în ../datasets/aptos_augmented_reorganized:

--- TRAIN ---
Clasa 0: 1434 imagini
Clasa 1: 300 imagini
Clasa 2: 808 imagini
Clasa 3: 154 imagini
Clasa 4: 234 imagini

--- VAL ---
Clasa 0: 172 imagini
Clasa 1: 40 imagini
Clasa 2: 104 imagini
Clasa 3: 22 imagini
Clasa 4: 28 imagini

--- TEST ---
Clasa 0: 199 imagini
Clasa 1: 30 imagini
Clasa 2: 87 imagini
Clasa 3: 17 imagini
Clasa 4: 33 imagini

